# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** A line chart is best because the question asks how a measure changes over time. I index each continent to 100 in 1970, so the lines compare proportional growth rather than raw population size. The steepest 2022 index identifies the fastest-growing continent.

In [ ]:
# Q1 — relative population growth by continent
continent_year = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = continent_year[continent_year['Year'] == 1970][['Continent', 'Population']].rename(columns={'Population': 'Population_1970'})
continent_year = continent_year.merge(base, on='Continent')
continent_year['Index (1970 = 100)'] = continent_year['Population'] / continent_year['Population_1970'] * 100
latest = continent_year[continent_year['Year'] == 2022].sort_values('Index (1970 = 100)', ascending=False)
print('Fastest-growing continent:', latest.iloc[0]['Continent'], '| index:', round(latest.iloc[0]['Index (1970 = 100)'], 1))
fig = px.line(continent_year, x='Year', y='Index (1970 = 100)', color='Continent',
              title='Africa’s population has grown fastest since 1970',
              color_discrete_sequence=px.colors.qualitative.Safe, template='plotly_white')
fig.update_layout(xaxis_title=None, yaxis_title='Population index (1970 = 100)', legend_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** A sorted horizontal bar chart makes the 15 largest absolute population gains easy to rank and compare. Colour distinguishes African countries from the rest, allowing the African count to be read directly.

In [ ]:
# Q2 — top 15 countries by absolute population gain
top15 = df.copy()
top15['Population added'] = top15['2022 Population'] - top15['1970 Population']
top15 = top15.nlargest(15, 'Population added').sort_values('Population added')
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
print('African countries in the top 15:', (top15['Continent'] == 'Africa').sum())
fig = px.bar(top15, x='Population added', y='Country', orientation='h', color='Group',
             title='India and China added the most people between 1970 and 2022',
             color_discrete_map={'Africa':'#2878B5', 'Other continents':'#9A9A9A'},
             template='plotly_white', text='Population added')
fig.update_traces(texttemplate='%{x:.2s}', textposition='outside', cliponaxis=False)
fig.update_layout(xaxis_title='Population added (people)', yaxis_title=None, legend_title=None)
fig.update_xaxes(showgrid=False)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** A scatter plot shows the relationship between land area and population for every country. Logarithmic axes make countries of very different sizes visible, while continent colour helps reveal regional patterns and outliers.

In [ ]:
# Q3 — land area versus 2022 population
fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', hover_name='Country',
                 log_x=True, log_y=True, title='Land area alone does not determine population size',
                 template='plotly_white', color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_traces(marker=dict(size=9, opacity=0.75))
fig.update_layout(xaxis_title='Land area (km², log scale)', yaxis_title='2022 population (log scale)', legend_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')
fig.show()
print('Largest populations:', df.nlargest(5, '2022 Population')[['Country','Continent','Area','2022 Population']].to_string(index=False))
print('Largest-area countries:', df.nlargest(5, 'Area')[['Country','Continent','Area','2022 Population']].to_string(index=False))

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** A box plot compares the distribution of population density across continents, showing medians, spread and outliers rather than only averages. A logarithmic y-axis handles the strong skew caused by very densely populated territories.

In [ ]:
# Q4 — distribution of population density by continent
fig = px.box(df, x='Continent', y='Density', color='Continent',
             title='Population density varies widely within every continent',
             template='plotly_white', points='outliers',
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(xaxis_title=None, yaxis_title='People per km² (log scale)', showlegend=False)
fig.update_yaxes(type='log', showgrid=True, gridcolor='#eeeeee')
fig.update_xaxes(showgrid=False)
fig.show()
print('Median density by continent:')
print(df.groupby('Continent')['Density'].median().sort_values(ascending=False).round(1))

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** A choropleth map is appropriate because the question asks where population growth and decline occurred geographically. The colour scale represents percentage change from 2000 to 2022, with negative values showing shrinking populations.

In [ ]:
# Q5 — percentage population change, 2000 to 2022
map_df = df.copy()
map_df['Population change (%)'] = (map_df['2022 Population'] / map_df['2000 Population'] - 1) * 100
fig = px.choropleth(map_df, locations='CCA3', locationmode='ISO-3', color='Population change (%)',
                    hover_name='Country', color_continuous_scale='RdYlBu',
                    color_continuous_midpoint=0,
                    title='Population growth is widespread, but several countries have shrunk',
                    template='plotly_white')
fig.update_layout(geo=dict(showframe=False, showcoastlines=False),
                  coloraxis_colorbar_title='Change (%)')
fig.show()
print('Largest percentage increases:')
print(map_df.nlargest(5, 'Population change (%)')[['Country','Population change (%)']].round(1).to_string(index=False))
print('Largest percentage decreases:')
print(map_df.nsmallest(5, 'Population change (%)')[['Country','Population change (%)']].round(1).to_string(index=False))

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** A treemap shows the hierarchy in one figure: continent-sized blocks represent total population and nested country blocks show each continent's contribution. Area is proportional to population, making the largest population centres immediately visible.

In [ ]:
# Q6 — world population hierarchy in 2022
tree = df.groupby(['Continent', 'Country'], as_index=False)['2022 Population'].sum()
fig = px.treemap(tree, path=['Continent', 'Country'], values='2022 Population',
                 title='Asia contains the largest share of the world’s population',
                 template='plotly_white', color='2022 Population',
                 color_continuous_scale='Blues')
fig.update_layout(margin=dict(t=60, l=10, r=10, b=10),
                  coloraxis_showscale=False)
fig.show()
print('Population by continent in 2022:')
print(df.groupby('Continent')['2022 Population'].sum().sort_values(ascending=False).to_string())

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** A 100% stacked bar chart compares the composition of the world population at each year. Each bar totals 100%, so changes in the Africa, Asia, Europe and rest-of-world segments show how their shares shifted over time.

In [5]:
# Q7 — population shares over time
shares = df_long.groupby(['Year', 'Continent'], as_index=False)['Population'].sum()
world_totals = shares.groupby('Year')['Population'].transform('sum')
shares['Share (%)'] = shares['Population'] / world_totals * 100
shares['Group'] = np.where(shares['Continent'].isin(['Africa', 'Asia', 'Europe']),
                           shares['Continent'], 'Rest of the world')
shares = shares.groupby(['Year', 'Group'], as_index=False)['Share (%)'].sum()
order = ['Africa', 'Asia', 'Europe', 'Rest of the world']
fig = px.bar(shares, x='Year', y='Share (%)', color='Group', barmode='stack',
             category_orders={'Group': order},
             title='Africa’s share is rising while Europe’s share is falling',
             color_discrete_map={'Africa':'#2878B5', 'Asia':'#8C8C8C',
                                 'Europe':'#E69F00', 'Rest of the world':'#BDBDBD'},
             template='plotly_white')
fig.update_layout(xaxis_title=None, yaxis_title='Share of world population (%)', legend_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(range=[0,100], showgrid=False)
fig.show()
print(shares.pivot(index='Year', columns='Group', values='Share (%)').round(2).to_string())